# 3. Use a trained model on leaf images

This notebook loads a saved SmallCNN, ResNet18, or EfficientNet-B0 checkpoint and predicts the class for one image or a folder of images. It does not train the model.

## Kaggle steps

1. Attach the selected checkpoint and your leaf image(s) as Kaggle inputs.
2. Set `CHECKPOINT_PATH` and `IMAGE_PATH` in the code cell.
3. Run the cell. It saves the top-five predictions to `/kaggle/working/predictions/predictions.csv`.

The checkpoint contains its architecture, ordered class labels, and normalization values. Keep the checkpoint from Notebook 1 or the Phase 2 SmallCNN run together with its metadata.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from tqdm.auto import tqdm

# Change these two values. Attach the model checkpoint and your leaf image(s) as Kaggle inputs.
CHECKPOINT_PATH = "/kaggle/input/your-model/selected_model_best.pt"
IMAGE_PATH = "/kaggle/input/your-leaf-images/leaf.jpg"  # One image or a folder containing images.
ARCHITECTURE_IF_MISSING = "smallcnn"  # Only needed for a legacy checkpoint without architecture metadata.
OUTPUT_DIR = Path("/kaggle/working/predictions")
IMAGE_SIZE=224; NUM_WORKERS=0
DEVICE=torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
checkpoint_path=Path(CHECKPOINT_PATH)
if not checkpoint_path.is_file(): raise FileNotFoundError("Set CHECKPOINT_PATH to an attached .pt checkpoint.")
checkpoint=torch.load(checkpoint_path,map_location="cpu",weights_only=False)
classes=checkpoint.get("classes")
if not classes: raise ValueError("Checkpoint does not contain class names. Use the Phase 2 SmallCNN checkpoint or a checkpoint from Notebook 1.")
architecture=checkpoint.get("architecture",ARCHITECTURE_IF_MISSING)

class SmallCNN(nn.Module):
    def __init__(self,n):
        super().__init__(); blocks=[]; inc=3
        for outc in (32,64,128,256): blocks += [nn.Conv2d(inc,outc,3,padding=1,bias=False),nn.BatchNorm2d(outc),nn.ReLU(inplace=True),nn.MaxPool2d(2)]; inc=outc
        self.features=nn.Sequential(*blocks); self.head=nn.Sequential(nn.AdaptiveAvgPool2d(1),nn.Flatten(),nn.Dropout(.30),nn.Linear(256,n))
    def forward(self,x): return self.head(self.features(x))

def build_model(name,n):
    if name=="smallcnn": return SmallCNN(n)
    if name=="resnet18":
        m=models.resnet18(weights=None); m.fc=nn.Linear(m.fc.in_features,n); return m
    if name=="efficientnet_b0":
        m=models.efficientnet_b0(weights=None); m.classifier[-1]=nn.Linear(m.classifier[-1].in_features,n); return m
    raise ValueError(f"Unknown architecture in checkpoint: {name}")

model=build_model(architecture,len(classes))
state=checkpoint.get("model_state_dict",checkpoint.get("state_dict"))
if state is None: raise ValueError("Checkpoint has no model weights.")
model.load_state_dict(state); model.to(DEVICE).eval()
mean=tuple(checkpoint.get("normalization_mean",[.5,.5,.5])); std=tuple(checkpoint.get("normalization_std",[.5,.5,.5]))
transform=transforms.Compose([transforms.Resize((IMAGE_SIZE,IMAGE_SIZE)),transforms.ToTensor(),transforms.Normalize(mean,std)])
source=Path(IMAGE_PATH)
extensions={".jpg",".jpeg",".png",".bmp",".webp"}
images=[source] if source.is_file() else sorted(p for p in source.rglob("*") if p.suffix.lower() in extensions)
if not images: raise FileNotFoundError("No supported images found. Set IMAGE_PATH to a JPG/PNG image or an input folder.")

class PredictImages(Dataset):
    def __len__(self): return len(images)
    def __getitem__(self,i):
        with Image.open(images[i]) as im: return transform(im.convert("RGB"))

loader=DataLoader(PredictImages(),batch_size=32,shuffle=False,num_workers=NUM_WORKERS)
rows=[]
with torch.inference_mode():
    offset=0
    for x in tqdm(loader,desc="Predicting"):
        x=x.to(DEVICE)
        if DEVICE.type=="cuda": x=x.contiguous(memory_format=torch.channels_last)
        with torch.amp.autocast(device_type=DEVICE.type,enabled=(DEVICE.type=="cuda")):
            probs=model(x).softmax(1).float().cpu().numpy()
        for j,p in enumerate(probs):
            top=np.argsort(p)[::-1][:5]
            rows.append({"image":str(images[offset+j]),"predicted_class":classes[int(top[0])],"confidence":float(p[top[0]]),
                         "top5":"; ".join(f"{classes[int(k)]}: {p[k]:.3f}" for k in top)})
        offset+=len(probs)
predictions=pd.DataFrame(rows)
predictions.to_csv(OUTPUT_DIR/"predictions.csv",index=False)
display(predictions.head(20))
print("Architecture:",architecture,"| device:",DEVICE,"| saved:",OUTPUT_DIR/"predictions.csv")
print("Confidence is the model's softmax score. It is not a calibrated probability that the diagnosis is correct.")
if len(images)==1:
    display(Image.open(images[0]).convert("RGB"))
